# 已选长期资料与刷新缺口复核

股票×交易日为价格记录单位。只查询原封存区间之外的价格；本笔记本自身仅读取已有审计输出、源码和日期日历，不读取价格、收益目标或封存标签。价格质量检查的完整可重跑代码为scripts/audit_earnings_history.py，日期切分代码为src/quant_research/earnings_walkforward.py。

八表相同的历史别名只计一次。三年训练、一年验证、下一季度的23个槽位只是开发日历；未构成事件队列、模型训练或账户收益。20个完整交易日必须严格位于拟合标签实际结束与下一分区之间，这一保守实现口径仍需正式登记。未知评价终点保留，跨封存终点禁止读取目标。

当前资料仍为reconstructed/training，行动、历史风险、身份、行业等资格仍未通过。缺因子有原隔离原因，不填1，也不删除后宣称全股票池完整。


In [1]:
import json, hashlib
from pathlib import Path
import pandas as pd

project = Path('/Users/guo/Documents/stocks/quant-research')
audit_root = project / 'artifacts/earnings-validation-20261012/history-audit-v4'
read = lambda p: json.loads(p.read_text())
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
audit = read(audit_root / 'audit.json')
for name, expected in audit['output_hashes'].items():
    assert sha(audit_root / name) == expected, name
for source in ['scripts/audit_earnings_history.py', 'src/quant_research/earnings_walkforward.py', 'configs/earnings-drift-validation-20261012-v1.json']:
    source_path = project / source
    assert sha(source_path) == audit['source_hashes'][str(source_path)], source
history = read(audit_root / 'historical-profile.json')
refresh = read(audit_root / 'refresh-profile.json')
folds = read(audit_root / 'development-folds.json')['folds']
print(json.dumps({'status': audit['status'], 'unique_long_history_table_sets': audit['unique_long_history_table_sets'], 'sealed_price_rows_queried': audit['sealed_price_rows_read'], 'formal_profit_qualified': audit['formal_profit_qualified']}, ensure_ascii=False))


{"status": "selected_history_and_refresh_profiled_no_return_experiment", "unique_long_history_table_sets": 1, "sealed_price_rows_queried": 0, "formal_profit_qualified": false}


In [2]:
# Independent quarter enumeration uses the source calendar only, without price/target columns.
from datetime import date
from calendar import monthrange
source = project / 'artifacts/full-market-training-20260910-v1'
calendar = pd.read_parquet(source / 'snapshot/calendar.parquet', columns=['date', 'is_open'])
seal = audit['sealed_holdout_start']
dates = sorted(set(calendar.loc[calendar.is_open.astype(bool), 'date']))
dates = [d for d in dates if history['inspected_first_date'] <= d <= history['inspected_last_date'] and d < seal]
assert dates == history['observed_dates']
expected = []
for year in range(date.fromisoformat(dates[0]).year + 4, date.fromisoformat(seal).year + 1):
    for quarter, month in enumerate([1, 4, 7, 10], start=1):
        begin = date(year, month, 1).isoformat()
        train = date(year - 4, month, 1).isoformat()
        stop_month = month + 2
        end = date(year, stop_month, monthrange(year, stop_month)[1]).isoformat()
        sessions = [d for d in dates if begin <= d <= end]
        if train < dates[0] or end >= seal or not sessions:
            continue
        last = dates.index(sessions[-1])
        if last + 21 >= len(dates):
            continue
        expected.append((f'{year}Q{quarter}', train, date(year - 1, month, 1).isoformat(), begin, end, dates[last + 21]))
actual = [(f['fold_id'], f['train_start'], f['validation_start'], f['test_start'], f['test_end'], f['normal_exit_observed_through']) for f in folds]
assert actual == expected
print(json.dumps({'independently_reproduced_calendar_slots': len(expected), 'first': expected[0][0], 'last': expected[-1][0], 'sample_endpoint_partitioning_applied': False, 'historical_return_experiment_run': False}, ensure_ascii=False))


{"independently_reproduced_calendar_slots": 23, "first": "2019Q4", "last": "2025Q2", "sample_endpoint_partitioning_applied": false, "historical_return_experiment_run": false}


In [3]:
# Profiles are retained source-check results, not investment outcomes.
assert history['duplicate_business_key_excess'] == 0
assert refresh['duplicate_business_key_excess'] == 0
assert sum(refresh['missing_factor_rows_by_reason'].values()) == refresh['nonfinite_factor_rows']
assert refresh['identified_main_board_missing_factor_rows'] == sum(refresh['missing_factor_rows_by_board'].get(board, 0) for board in ['shanghai_main', 'shenzhen_main'])
print(json.dumps({'historical_rows_inspected': history['inspected_rows'], 'historical_dates': [history['inspected_first_date'], history['inspected_last_date']], 'historical_nonfinite_required_rows': history['rows_with_nonfinite_required_field'], 'historical_multiple_sequence_instruments': history['instruments_with_multiple_sequence_ids'], 'refresh_rows_inspected': refresh['inspected_rows'], 'refresh_dates': [refresh['inspected_first_date'], refresh['inspected_last_date']], 'refresh_missing_factor_rows': refresh['nonfinite_factor_rows'], 'refresh_missing_factor_share': refresh['nonfinite_factor_share'], 'identified_main_board_missing_factor_rows': refresh['identified_main_board_missing_factor_rows'], 'identified_main_board_missing_factor_share': refresh['identified_main_board_missing_factor_rows'] / refresh['identified_main_board_rows'], 'reasons': refresh['missing_factor_rows_by_reason']}, ensure_ascii=False, indent=2))


{
  "historical_rows_inspected": 10016834,
  "historical_dates": [
    "2015-09-01",
    "2025-08-06"
  ],
  "historical_nonfinite_required_rows": 0,
  "historical_multiple_sequence_instruments": 189,
  "refresh_rows_inspected": 227572,
  "refresh_dates": [
    "2026-08-07",
    "2026-10-12"
  ],
  "refresh_missing_factor_rows": 6406,
  "refresh_missing_factor_share": 0.028149332958360432,
  "identified_main_board_missing_factor_rows": 3906,
  "identified_main_board_missing_factor_share": 0.029866495389273752,
  "reasons": {
    "reference_change_requires_action_evidence": 3082,
    "missing_cross_provider_anchor": 148,
    "missing_session_no_suspension_assumption": 126,
    "missing_anchor_factor": 3050
  }
}
